In [5]:
import numpy as np                                          
from scipy.stats import norm, bernoulli, norm as scipy_norm 
import pandas as pd                                         
from tqdm.notebook import tqdm                              
%load_ext autoreload
%autoreload 2                                               

from utils.inference import (
    train_sampling_rule,          # fits XGBoost to predict LLM squared error from demo features
    sampling_rule_predict,        # applies fitted rule to get per-observation error predictions
    perspective_driven_inference, # PDI estimator + bootstrap CI
)
from ppi_py import ppi_mean_pointestimate, ppi_mean_ci      # PPI++ point estimate and CI

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
alpha        = 0.05   # significance level → 95% confidence intervals
burnin_steps = 150    # first 150 obs collected uniformly to bootstrap the sampling rule
n_batches    = 5      # number of adaptive batches after burn-in
n_human      = 800    # total human annotation budget (burn-in + adaptive batches)
N            = None   # placeholder; set after data loading
random_state = 42     # global random seed for numpy and XGBoost
n_trials     = 10     # number of independent trial repetitions per model
tau          = 0.1    # smoothing weight: mixes adaptive probs with uniform within each batch
                      #   prevents zero-probability allocations for rare groups
q_power      = 1.0    # exponent on predicted error when computing sampling probs
                      #   1.0 = raw predicted error (moderate concentration)
                      #   0.5 = square root (softer); 2.0 = squared (more aggressive)

polite_threshold = 4  # binary label: polite = rating >= 4 on a 1–5 scale

# demographic columns used as features for the sampling rule
demo_cols = ['gender', 'race', 'age', 'education']

# 10 target demographic groups across gender, race, and education
# each value is a boolean mask function: data -> array of True/False
demographic_groups = {
    'Woman':            lambda df: df['gender'] == 'Woman',
    'Man':              lambda df: df['gender'] == 'Man',
    'Non-binary':       lambda df: df['gender'] == 'Non-binary',
    'White':            lambda df: df['race'] == 'White',
    'Black/Afr. Am.':   lambda df: df['race'] == 'Black or African American',
    'Hispanic/Latino':  lambda df: df['race'] == 'Hispanic or Latino',
    'Asian':            lambda df: df['race'] == 'Asian',
    'College degree':   lambda df: df['education'] == 'College degree',
    'HS diploma':       lambda df: df['education'] == 'High school diploma or equivalent',
    'Grad. degree':     lambda df: df['education'] == 'Graduate degree',
}

# IPW weighted mean: sum(y * w) / sum(w), ignoring NaN entries
# used inside perspective_driven_inference as the base estimator
def mean_estimator(y, weights):
    y, weights = y[~np.isnan(y)], weights[~np.isnan(y)]  # drop unobserved (NaN) entries
    return np.sum(y * weights) / np.sum(weights)

# ordered list of group names (used to look up group indices by name)
group_names = list(demographic_groups.keys())


In [7]:
# 8 LLMs to evaluate; each has zero-shot, few-shot, and persona-prompt columns in the CSV
models = [
    'gpt-5.2',
    'claude-sonnet-4.6',
    'claude-opus-4.6',
    'gemini-3.1-pro',
    'claude-haiku-4.5',
    'llama-3-8b',
    'mistral-large-2512',
    'gpt-oss-120b',
]

# map each model to its three CSV column names
model_cols = {
    m: {
        'zs': f'{m} (zero shot prompting)',
        'fs': f'{m} (few shot)',
        'pp': f'{m} (persona prompt)',
    }
    for m in models
}

# keep only rows where every model has a prediction (drops ~7% of rows)
required_cols = [c for m in models for c in model_cols[m].values()]
data_mm = (
    pd.read_csv('data/raw_data_llm_politeness.csv')
      .dropna(subset=required_cols)           # remove rows with any missing LLM prediction
      .sample(frac=1, random_state=random_state)  # shuffle so burn-in window is demographically mixed
      .reset_index(drop=True)
      .copy()
)
# binary human label: 1 = polite (rating >= 4), 0 = not polite
data_mm['human_true'] = (data_mm['politeness'] >= polite_threshold).astype(float)
N_mm = len(data_mm)  # total number of (text, annotator) pairs

# one-hot encode the 4 demographic columns → sampling rule feature matrix
# shape: (N_mm, n_demo_categories); each row is the demographic profile of one annotator
demo_features_mm = pd.get_dummies(data_mm[demo_cols]).astype(float).values

# boolean mask for each of the 10 demographic groups, shape (N_mm,) each
group_masks_mm = [mask_fn(data_mm).values for mask_fn in demographic_groups.values()]

# true_thetas_mm[g] = true mean of human_true within group g (the estimand)
true_thetas_mm = np.array([data_mm.loc[m, 'human_true'].mean() for m in group_masks_mm])

# expected number of annotations each group receives under uniform sampling
group_sizes_mm = np.array([m.sum() for m in group_masks_mm])
expected_n_mm  = n_human * group_sizes_mm / N_mm

# theoretical variance of the human-only estimator per group: theta*(1-theta)/E[n_g]
# used as the numerator in the ESS gain formula
var_human_mm = true_thetas_mm * (1 - true_thetas_mm) / expected_n_mm

# summary metrics are averaged over these 3 gender groups only
gender_names_mm = ['Woman', 'Man', 'Non-binary']
gender_idx_mm   = [group_names.index(n) for n in gender_names_mm]  # integer indices into group_masks_mm

In [8]:
def run_model_trials(Hhat_best, Hhat_zs, Hhat_fs, Hhat_pp, data, demo_features,
                     group_masks, true_thetas):
    """Run PDI/PPI/LLM-only trials for one model.

    Hhat_best : LLM predictions used for PPI/PDI (best-coverage variant, chosen in the run cell)
    Hhat_zs/fs/pp : all three prompting variants, shown as LLM-only baselines in the table
    """
    n_groups = len(group_masks)   # 10 demographic groups
    N_loc    = len(data)          # number of (text, annotator) pairs

    # accumulators: shape (n_trials, n_groups); NaN = not yet computed
    methods  = ['ZS LLM', 'FS LLM', 'PP LLM', 'PPI', 'PDI']
    ests     = {m: np.full((n_trials, n_groups), np.nan) for m in methods}  # point estimates
    covered  = {m: np.full((n_trials, n_groups), np.nan) for m in methods}  # 1 if CI covers true theta

    batch_size_loc = (N_loc - burnin_steps) // n_batches  # observations per adaptive batch
    z_crit = scipy_norm.ppf(1 - alpha / 2)               # 1.96 for 95% CI (LLM-only baselines)

    # apply q_power exponent to raw XGBoost predictions to get sampling scores
    def compute_q_loc(rule, features):
        return np.clip(sampling_rule_predict(rule, features), 1e-8, None) ** q_power

    for trial in range(n_trials):
        np.random.seed(random_state + trial * 13)  # unique seed per trial → reproducible

        # ── uniform random sample (shared by PPI++ and LLM-only baselines) ──
        ppi_idx      = np.random.choice(N_loc, size=n_human, replace=False)  # n_human uniform indices
        ppi_selected = np.zeros(N_loc, dtype=bool)
        ppi_selected[ppi_idx] = True

        # ── PDI: demographic-adaptive sampling ───────────────────────────────
        H  = np.full(N_loc, np.nan)  # revealed human labels; NaN = not yet observed
        SP = np.zeros(N_loc)         # sampling probabilities π_i
        SD = np.zeros(N_loc)         # sampling decisions ξ_i (1 = annotated)

        # burn-in: treat first burnin_steps obs as observed with probability 1
        H[:burnin_steps]  = data['human_true'].values[:burnin_steps]
        SP[:burnin_steps] = 1.0
        SD[:burnin_steps] = 1.0

        # fit initial sampling rule on burn-in data:
        # features = one-hot annotator demographics; labels = squared LLM error
        sampling_rule = train_sampling_rule(
            demo_features[:burnin_steps],
            (H[:burnin_steps] - Hhat_best[:burnin_steps]) ** 2,
            seed=random_state)
        q = compute_q_loc(sampling_rule, demo_features)  # predicted sampling score for all N obs

        for b in range(n_batches):
            # index range for this batch
            batch_inds = (
                np.arange(burnin_steps + b * batch_size_loc,
                          burnin_steps + (b + 1) * batch_size_loc)
                if b < n_batches - 1
                else np.arange(burnin_steps + b * batch_size_loc, N_loc)  # last batch takes remainder
            )
            remaining = n_human - int(SD.sum())  # annotations left in budget
            if remaining <= 0:
                break
            # split remaining budget roughly evenly across remaining batches
            target = min(round(remaining / (n_batches - b)), remaining, len(batch_inds))

            # normalize q scores within batch into a probability distribution
            p = q[batch_inds];  p = p / p.sum()
            # smooth with uniform: (1-tau)*adaptive + tau*uniform, then re-normalize
            p = (1 - tau) * p + tau / len(batch_inds);  p = p / p.sum()

            # fixed-size sample of exactly `target` indices (no Bernoulli variance)
            sel = np.random.choice(len(batch_inds), size=target, replace=False, p=p)

            # reveal human labels for selected observations
            H[batch_inds[sel]] = data['human_true'].values[batch_inds[sel]]

            # store sampling decisions and probabilities for this batch
            SD[batch_inds] = 0.0;  SD[batch_inds[sel]] = 1.0        # ξ_i
            SP[batch_inds] = np.clip(target * p, 1e-4, 1.0)          # π_i = target * p_i

            # update sampling rule using all annotations collected so far (except last batch)
            if b < n_batches - 1:
                lab = np.where(~np.isnan(H))[0]  # all annotated indices so far
                sampling_rule = train_sampling_rule(
                    demo_features[lab], (H[lab] - Hhat_best[lab]) ** 2,
                    seed=random_state)
                q = compute_q_loc(sampling_rule, demo_features)

        # ── evaluate all methods per demographic group ────────────────────────
        for g, (mask, true_theta) in enumerate(zip(group_masks, true_thetas)):
            # observations in this group that were uniformly sampled (for PPI / LLM-only)
            llm_mask    = mask & ppi_selected
            g_sampled   = mask & ppi_selected   # labeled subset for PPI
            g_unsampled = mask & ~ppi_selected  # unlabeled subset for PPI

            # ── LLM-only baselines: treat LLM mean over uniform sample as estimate ──
            for lm_key, Hhat_lm in [('ZS LLM', Hhat_zs),
                                     ('FS LLM', Hhat_fs),
                                     ('PP LLM', Hhat_pp)]:
                if llm_mask.sum() >= 2:
                    lm_est = Hhat_lm[llm_mask].mean()  # mean LLM prediction over sampled group members
                    # normal approximation CI using Bernoulli SE
                    se = np.sqrt(max(lm_est * (1 - lm_est), 1e-6) / llm_mask.sum())
                    ests[lm_key][trial, g]    = lm_est
                    covered[lm_key][trial, g] = int(
                        lm_est - z_crit * se <= true_theta <= lm_est + z_crit * se)

            # ── PPI++: rectified estimator with uniform sampling and optimal lambda ──
            if g_sampled.sum() >= 2 and g_unsampled.sum() >= 1:
                Y_lab      = data['human_true'].values[g_sampled]   # observed human labels
                Yhat_lab   = Hhat_best[g_sampled]                   # LLM predictions for labeled obs
                Yhat_unlab = Hhat_best[g_unsampled]                 # LLM predictions for unlabeled obs
                est    = ppi_mean_pointestimate(Y_lab, Yhat_lab, Yhat_unlab, lam=None)  # lam=None → PPI++
                lb, ub = ppi_mean_ci(Y_lab, Yhat_lab, Yhat_unlab, alpha=alpha, lam=None)
                ests['PPI'][trial, g]    = est
                covered['PPI'][trial, g] = int(lb <= true_theta <= ub)

            # ── PDI: rectified estimator with adaptive sampling and optimal lambda ──
            if SD[mask].sum() >= 2:  # need at least 2 annotated obs in this group
                est, (lb, ub) = perspective_driven_inference(
                    mean_estimator,
                    Y=H[mask],                   # human labels for this group (NaN = not annotated)
                    Yhat=Hhat_best[mask],         # LLM predictions for this group
                    sampling_probs=SP[mask],      # π_i: probability each obs was selected
                    sampling_decisions=SD[mask],  # ξ_i: whether each obs was actually annotated
                    alpha=alpha, lam=None,        # lam=None → optimise lambda via bootstrap
                    n_resamples=100, n_resamples_lam=20)  # bootstrap samples for CI and lambda
                ests['PDI'][trial, g]    = est
                covered['PDI'][trial, g] = int(lb <= true_theta <= ub)

    return ests, covered


In [9]:
model_results_mm = {}  # stores final results keyed by model name

# maps variant name → key used in ests/covered dicts
variant_key = {'zero shot': 'ZS LLM', 'few shot': 'FS LLM', 'persona': 'PP LLM'}

for model in tqdm(models, desc='Models'):
    cols = model_cols[model]

    # skip model if any of its columns are missing from the CSV
    missing = [c for c in cols.values() if c not in data_mm.columns]
    if missing:
        print(f'Skipping {model}: missing {missing}')
        continue

    # binarize LLM predictions: 1 = predicts polite, 0 = predicts not polite
    Hhat_zs = (data_mm[cols['zs']] >= polite_threshold).astype(float).to_numpy()
    Hhat_fs = (data_mm[cols['fs']] >= polite_threshold).astype(float).to_numpy()
    Hhat_pp = (data_mm[cols['pp']] >= polite_threshold).astype(float).to_numpy()
    human   = data_mm['human_true'].values

    variants = {'zero shot': Hhat_zs, 'few shot': Hhat_fs, 'persona': Hhat_pp}

    # run trials for all 3 variants; LLM-only coverage is used to select the best one
    # coverage is averaged over the 3 gender groups (Woman, Man, Non-binary)
    all_results = {}
    for vname, Hhat_v in variants.items():
        e, c = run_model_trials(
            Hhat_v, Hhat_zs, Hhat_fs, Hhat_pp,
            data_mm, demo_features_mm, group_masks_mm, true_thetas_mm)
        # LLM-only coverage for this variant: fraction of trials where CI covers true theta,
        # averaged over the 3 gender groups
        llm_cov = np.nanmean([np.nanmean(c[variant_key[vname]][:, i]) for i in gender_idx_mm])
        all_results[vname] = {'ests': e, 'covered': c, 'llm_cov': llm_cov}

    # pick the variant whose LLM-only CI has the highest coverage as the predictor for PPI/PDI
    best_variant = max(all_results, key=lambda v: all_results[v]['llm_cov'])

    model_results_mm[model] = {
        'ests':         all_results[best_variant]['ests'],     # point estimates (n_trials x n_groups)
        'covered':      all_results[best_variant]['covered'],  # coverage indicators (n_trials x n_groups)
        'best_variant': best_variant,                          # which prompting strategy was selected
        # overall LLM error rates (fraction of all N obs where LLM != human)
        'llm_err_zs':   (Hhat_zs != human).mean(),
        'llm_err_fs':   (Hhat_fs != human).mean(),
        'llm_err_pp':   (Hhat_pp != human).mean(),
        # LLM-only coverage per variant (used for best-variant selection)
        'llm_cov_zs':   all_results['zero shot']['llm_cov'],
        'llm_cov_fs':   all_results['few shot']['llm_cov'],
        'llm_cov_pp':   all_results['persona']['llm_cov'],
    }

print('Done.')


Models:   0%|          | 0/8 [00:00<?, ?it/s]

Done.


In [10]:
mw  = 25                       # column width for method label
sep = '=' * (mw + 26)          # separator line
sub = '-' * (mw + 26)          # sub-separator line
ci_level_mm = int((1 - alpha) * 100)  # 95

variant_key = {'zero shot': 'ZS LLM', 'few shot': 'FS LLM', 'persona': 'PP LLM'}

for model in models:
    if model not in model_results_mm:
        continue
    res          = model_results_mm[model]
    ests_m       = res['ests']
    cov_m        = res['covered']
    best_variant = res['best_variant']  # prompting strategy used for PPI/PDI
    best_key     = variant_key[best_variant]

    # empirical variance of point estimates across 10 trials, per gender group
    var_ppi = np.array([np.nanvar(ests_m['PPI'][:, i], ddof=1) for i in gender_idx_mm])
    var_pdi = np.array([np.nanvar(ests_m['PDI'][:, i], ddof=1) for i in gender_idx_mm])

    # ESS gain = (Var_human / Var_method - 1) * 100%, averaged over gender groups
    # positive = method is more efficient than human-only at same budget
    ppi_ess = np.nanmean(np.where(var_ppi > 0, var_human_mm[gender_idx_mm] / var_ppi - 1, np.nan) * 100)
    pdi_ess = np.nanmean(np.where(var_pdi > 0, var_human_mm[gender_idx_mm] / var_pdi - 1, np.nan) * 100)

    # mark the best LLM variant with ★ in the LLM-only rows
    zs_label = f'{model} (zero shot)' + (' ★' if best_variant == 'zero shot' else '')
    fs_label = f'{model} (few shot)'  + (' ★' if best_variant == 'few shot'  else '')
    pp_label = f'{model} (persona)'   + (' ★' if best_variant == 'persona'   else '')

    # rows: (display label, internal key, has_ess, [ess_value])
    rows = [
        (zs_label, 'ZS LLM', False),
        (fs_label, 'FS LLM', False),
        (pp_label, 'PP LLM', False),
        (f'PPI  [★ {best_variant}]', 'PPI', True, ppi_ess),  # PPI++ with best variant
        (f'PDI  [★ {best_variant}]', 'PDI', True, pdi_ess),  # PDI  with best variant
    ]

    print(f'Coverage & ESS Gain  (target {ci_level_mm}%,  n_human={n_human},  {n_trials} trials)')
    print(f'Averaged across: {gender_names_mm}')  # Woman, Man, Non-binary
    print(sep)
    print(f'{"Method":<{mw}} {"Avg Coverage":>12} {"Avg ESS Gain":>12}')
    print(sub)
    for row in rows:
        label, key = row[0], row[1]
        has_ess    = row[2]
        # average fraction of trials where CI covered true theta, across 3 gender groups
        cov_avg = np.nanmean([np.nanmean(cov_m[key][:, i]) for i in gender_idx_mm])
        if has_ess:
            ess_str = f'{row[3]:>+11.1f}%'
        else:
            ess_str = f"{'N/A':>12}"  # ESS not defined for LLM-only (no human budget)
        print(f'{label:<{mw}} {cov_avg:>12.3f} {ess_str}')
    print(sep)
    print()


Coverage & ESS Gain  (target 95%,  n_human=800,  10 trials)
Averaged across: ['Woman', 'Man', 'Non-binary']
Method                    Avg Coverage Avg ESS Gain
---------------------------------------------------
gpt-5.2 (zero shot)              0.533          N/A
gpt-5.2 (few shot) ★             0.933          N/A
gpt-5.2 (persona)                0.733          N/A
PPI  [★ few shot]                0.867       +16.8%
PDI  [★ few shot]                0.967       +33.7%

Coverage & ESS Gain  (target 95%,  n_human=800,  10 trials)
Averaged across: ['Woman', 'Man', 'Non-binary']
Method                    Avg Coverage Avg ESS Gain
---------------------------------------------------
claude-sonnet-4.6 (zero shot)        0.267          N/A
claude-sonnet-4.6 (few shot)        0.400          N/A
claude-sonnet-4.6 (persona) ★        0.533          N/A
PPI  [★ persona]                 0.900       +16.9%
PDI  [★ persona]                 0.933       +47.5%

Coverage & ESS Gain  (target 95%,  n_human=